# Squishy Particle Simulator — Reproduce Paper Results

This notebook reproduces all figures and benchmarks from the EPD methods paper
(*Elastic Particle DEM: a capsule-shell model with tunable squishiness*).

**Structure**
| Section | Paper §§ | Scripts |
|---------|----------|---------|
| 0 — Setup | — | clone + install |
| 1 — Elastic calibration | §6–7 | `calibration_sweep`, `fixed_b_q_sweep`, `R0_spot_check` |
| 2 — Contact force law | §8 | `contact_law_fd`, `contact_law_extended` |
| 3 — Emulsion droplet model | §9 | `emulsion_benchmarks` (C, D, E) |
| 4 — Viscous drag | §10 | `test_E_stokes_drag`, `test_E_figures`, `test_E_movies` |

All scripts use the **v1 API**: `ParticleSpec`, `System`, `System.run(n, sample_every, callback)`.

> **GPU note:** simulations are fast on GPU (Colab T4/A100). CPU runs are correct but slow.

---
## Section 0 — Setup
Run once per session.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/KD-physics/Squishy-Particle-Simulator.git'

if not os.path.exists('Squishy-Particle-Simulator'):
    !git clone {REPO_URL}
%cd Squishy-Particle-Simulator/python_v2

!pip install -q numpy scipy matplotlib imageio tensorflow
!pip install -q .    # builds C++ candidacy-manager extension

print('Setup complete.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import src.simulation.tf_sim as tf_sim_mod
tf_sim_mod.set_dtype(tf.float64)

from src.epd.particles import ParticleSpec
from src.epd.objects  import Wall, OscillatingWall
from src.epd.system   import System
from src.epd.motion   import MotionSpec

print('v1 API imports OK')
print('TF version:', tf.__version__)

---
## Section 1 — Elastic Calibration (Paper §6–7)

The squishiness knob **ν** (0.18–0.94) is calibrated by sweeping the area-spring ratio **q**
and measuring the effective Poisson ratio from a two-disk quasi-static squeeze.

Key scripts:
- `calibration_sweep.py` — full q × N sweep → `results/calibration_sweep/calibration_data.json`
- `fixed_b_q_sweep.py` — fixed b=0.2, squishiness axis figure
- `R0_spot_check.py` — verifies R0 is a pure length scale (CV < 1%)

All use `run_squeeze_raw(q, tau, N, ...)` which builds a `System` internally.

In [ ]:
# Quick demo: single (q, N) squeeze evaluation via v1 API
from src.validation.twodisk_squeeze import run_squeeze_raw, interpolate_at_strain

TAU = np.sqrt(12.0 * 0.2)   # b=0.2 working point

frames = run_squeeze_raw(
    q=2.0, tau=TAU, N=48, R0=1.0,
    delta_max=0.10, n_frames=10,
    C_factor=3000.0, alpha_damp=2.0,
    strain_rate_ratio=0.001, verbose=False,
)
nu_meas, dA_frac = interpolate_at_strain(frames, eps_ref=0.08)

print(f'q=2.0  →  ν_meas = {nu_meas:.3f}  ΔA = {dA_frac*100:.2f}%')
print(f'(rubber-like: soft, incompressible)')

frames2 = run_squeeze_raw(
    q=0.1, tau=TAU, N=48, R0=1.0,
    delta_max=0.10, n_frames=10,
    C_factor=3000.0, alpha_damp=2.0,
    strain_rate_ratio=0.001, verbose=False,
)
nu_meas2, dA_frac2 = interpolate_at_strain(frames2, eps_ref=0.08)
print(f'q=0.1  →  ν_meas = {nu_meas2:.3f}  ΔA = {dA_frac2*100:.2f}%')
print(f'(glass-like: stiff, compressible)')

In [ ]:
# Full calibration sweep — generates calibration_data.json (GPU: ~5 min)
# Uncomment to run:
# %run src/validation/calibration_sweep.py
print('To run full sweep:')
print('  %run src/validation/calibration_sweep.py')
print('Output: results/calibration_sweep/calibration_data.json')
print()
print('Then run the fixed-b sweep and R0 check:')
print('  %run src/validation/fixed_b_q_sweep.py')
print('  %run src/validation/R0_spot_check.py')

---
## Section 2 — Contact Force Law (Paper §8)

The disk–disk contact force F(δ) follows a power law F ∝ δⁿ where n depends on ν.
The N-convergence shows n → n∞ as 1/N → 0.

Key scripts:
- `contact_law_fd.py` — F(δ) curves and power-law exponent vs. ν
- `contact_law_extended.py` — N-convergence for N ∈ {32, 48, 72, 120, 240}

**Requires:** `calibration_data.json` from Section 1.

In [ ]:
# Quick demo: F(δ) extraction from a squeeze run
from src.validation.contact_law_fd import _fd_from_frames, fit_power_law
from src.validation.twodisk_squeeze import run_squeeze_raw

TAU = np.sqrt(12.0 * 0.2)
R0, N = 1.0, 32

frames = run_squeeze_raw(
    q=2.0, tau=TAU, N=N, R0=R0,
    delta_max=0.10, n_frames=30,
    C_factor=3000.0, alpha_damp=2.0,
    strain_rate_ratio=0.001, verbose=False,
)
res = _fd_from_frames(frames, R0=R0, N=N)
A, n_exp = fit_power_law(res['delta'], res['F_dd'])

print(f'q=2.0 (rubber-like): F ∝ δ^{n_exp:.3f}')
print(f'  (analytic Hertz limit for 2D: n=0.5)')

if A is not None:
    fig, ax = plt.subplots(figsize=(5, 4))
    mask = (res['F_dd'] > 0) & (res['delta'] > 0)
    ax.loglog(res['delta'][mask] / R0, res['F_dd'][mask], 'b.', ms=3, label='simulation')
    d_fit = np.linspace(res['delta'][mask].min(), res['delta'][mask].max(), 50)
    ax.loglog(d_fit / R0, A * d_fit**n_exp, 'r--', lw=1.5, label=f'fit: n={n_exp:.2f}')
    ax.set_xlabel('δ / R₀'); ax.set_ylabel('F'); ax.set_title('F(δ) log–log')
    ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Full N-convergence runs (GPU: ~15 min)
# Requires calibration_data.json from Section 1
print('To run full contact-law suite (after calibration_sweep.py):')
print('  %run src/validation/contact_law_fd.py')
print('  %run src/validation/contact_law_extended.py')
print('Outputs: results/contact_law_fd/')

---
## Section 3 — Emulsion Droplet Model (Paper §9)

Three canonical benchmarks validate the emulsion model:
- **C** — Capillary-wave damping: mode-2 frequency ω₂ vs. analytic √6
- **D** — T1-event: centre droplet squeezed through outer pair (descending wall via v1 `Wall+MotionSpec`)
- **E** — Falling droplet: trajectory under gravity for multiple Bond numbers

All three use the v1 API (`ParticleSpec`, `System`, `System.run()`).
Benchmark D re-pins outer CMs between `System.run()` chunks rather than inside a callback.

In [ ]:
# Benchmark C: capillary-wave damping (12 periods, α=0.5)
from src.validation.emulsion_benchmarks import (
    run_capwave, _measure_omega, omega2_analytic, T2_analytic, tau0
)

ts, ells = run_capwave(alpha_damp=0.5, n_periods=12)   # 12 periods (matches benchmark protocol)
omega2_meas = _measure_omega(ts, ells)

print(f'ω₂ analytic = {omega2_analytic:.4f} rad/τ₀  (= √8; purely-radial 2D mode)')
if omega2_meas:
    err = abs(omega2_meas - omega2_analytic) / omega2_analytic
    print(f'ω₂ measured = {omega2_meas:.4f} rad/τ₀  (error {err*100:.1f}%)')
else:
    print('ω₂ measured = not enough crossings in 2 periods (expected for α=0.5)')

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(ts / tau0, ells, lw=1.5)
ax.axhline(0, color='gray', lw=0.5, ls='--')
ax.set_xlabel('t / τ₀'); ax.set_ylabel('Ellipticity')
ax.set_title('Mode-2 capillary-wave damping (α=0.5, 12 periods)')
plt.tight_layout(); plt.show()

In [ ]:
# Benchmark E quick test: single falling droplet (Bo=0.05, short run)
from src.validation.emulsion_benchmarks import run_single_fall

res = run_single_fall(g=0.05, t_max_tau=20.0)   # 20 τ₀ only
print(f'Bo=0.05: y_cm at t=20τ₀ = {res["ys"][-1]:.3f}')

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(res['ts'], res['ys'], lw=1.5, color='#ff7f0e', label='Bo=0.05')
ax.axhline(1.0, color='sienna', ls=':', lw=1, label='floor + R₀')
ax.set_xlabel('t'); ax.set_ylabel('y_cm')
ax.set_title('Falling emulsion droplet (v1 API, N=60)')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Full emulsion benchmarks (GPU: ~30 min for all three)
print('To run all emulsion benchmarks:')
print('  %run src/validation/emulsion_benchmarks.py --benchmarks C D E')
print()
print('Individual benchmarks:')
print('  %run src/validation/emulsion_benchmarks.py --benchmarks C')
print('  %run src/validation/emulsion_benchmarks.py --benchmarks D')
print('  %run src/validation/emulsion_benchmarks.py --benchmarks E')
print('Outputs: papers/summary_of_methods/figures/')

---
## Section 4 — Viscous Drag (Paper §10)

Four benchmarks test the Stokes-drag (`Oh` = Ohnesorge number) and background-flow features:
- **E1** — Terminal velocity: v_t = g / (2·Oh·ω₀²) for emulsion; elastic analogous
- **E2** — Background flow tracking: CM → U_bg exponentially with τ_drag
- **E3** — Shear flow: CM drifts, particle rotates at Ω ≈ Γ/2
- **E4** — Post-init `set_param('Oh', ...)` update, confirms new terminal velocity

Paper figures: terminal velocity vs. Oh sweep; flow-tracking time series.

In [ ]:
# Quick E1: terminal velocity (v1 API inline demo)
from src.epd.particles import ParticleSpec
from src.epd.objects  import Wall
from src.epd.system   import System
import numpy as np
import tensorflow as tf

Oh, g, N_ = 0.25, 0.05, 36
LX, LY = 10.0, 30.0

sys_ = System(LX, LY, periodic_x=False, periodic_y=False, g=g)
sys_.add_object(Wall((0, 0), (LX, 0), normal=(0, 1)))   # floor
sys_.add_particles(ParticleSpec(count=1, type='emulsion',
                                gamma=1.0, kappa=0.02, N_nodes=N_, Oh=Oh))
sys_.initialize(phi_target=0.80, seed=0, verbose=False,
                relax_only=True, n_relax_init=0)

# Place the droplet near the top of the box so it has free-fall room
# to reach terminal velocity before contacting the floor.
y_start = 0.7 * LY
shift = np.array([[0.0, y_start - sys_._state['x_cm'].numpy()[0, 1]]])
sys_._state['x_all'] = tf.constant(sys_._state['x_all'].numpy() + shift, dtype=tf.float64)
sys_._state['x_cm']  = tf.constant(sys_._state['x_cm'].numpy()  + shift, dtype=tf.float64)

ys = []
def _cb(s):
    snap = s.snapshot()
    ys.append(float(snap['x_cm'][0, 1]))
    return {}

dt = sys_._dt
n_steps  = int(40.0 / dt)   # 40 τ₀
s_every  = max(1, int(0.5 / dt))
sys_.run(n_steps, sample_every=s_every, callback=_cb, record_initial=True, verbose=False)

# Analytic terminal velocity for emulsion (paper §10): v_t = g / (2·Oh)
v_term_analytic = g / (2 * Oh)

# Estimate measured terminal velocity from last 10 sampled points (before any
# floor contact): take dy/dt over a window where the droplet is still falling.
ys_arr = np.array(ys)
ts_arr = np.arange(len(ys_arr)) * s_every * dt
dys    = np.gradient(ys_arr, ts_arr)
# Use the central window (mid-fall) for a clean estimate
mid    = slice(len(ys_arr)//3, 2*len(ys_arr)//3)
v_term_meas = float(np.mean(np.abs(dys[mid])))

print(f'Oh={Oh}, g={g}')
print(f'  y_cm(t=0)        = {ys[0]:.3f}')
print(f'  y_cm(t_final)    = {ys[-1]:.3f}')
print(f'  v_t analytic     = g/(2·Oh) = {v_term_analytic:.4f}')
print(f'  v_t measured     = {v_term_meas:.4f}')

In [ ]:
# Full E1–E4 benchmarks and paper figures
print('Run all drag benchmarks:')
print('  %run src/validation/test_E_stokes_drag.py')
print()
print('Generate paper figures:')
print('  %run src/validation/test_E_figures.py')
print('  → papers/summary_of_methods/figures/drag_terminal_velocity.png')
print('  → papers/summary_of_methods/figures/drag_flow_tracking.png')
print()
print('Render supplemental movies S12–S13:')
print('  %run src/validation/test_E_movies.py')
print('  → papers/summary_of_methods/movies/S12_terminal_velocity_drag.gif')
print('  → papers/summary_of_methods/movies/S13_shear_rotation.gif')

---
## Summary — All Paper Scripts

Run the full pipeline (on GPU, ~1 hour total):

```bash
# § 6–7: Elastic calibration
python src/validation/calibration_sweep.py
python src/validation/fixed_b_q_sweep.py
python src/validation/R0_spot_check.py

# § 8: Contact force law
python src/validation/contact_law_fd.py
python src/validation/contact_law_extended.py

# § 9: Emulsion model
python src/validation/emulsion_benchmarks.py --benchmarks C D E

# § 10: Viscous drag
python src/validation/test_E_stokes_drag.py
python src/validation/test_E_figures.py
python src/validation/test_E_movies.py
```

Outputs land in:
- `results/calibration_sweep/` — calibration tables and plots
- `results/contact_law_fd/` — F(δ) convergence
- `papers/summary_of_methods/figures/` — all paper figures
- `papers/summary_of_methods/movies/` — supplemental animations

All scripts use the **v1 API**: `ParticleSpec`, `System`, `System.run(n, sample_every, callback)`.